# Fitness Activity Data Analysis
Goal: analyze 200 days of synthetic fitness data: statistics, activity by day of week, the correlation between steps and calories, charts, and outliers in steps.

## 1. Create the dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
fitness = pd.DataFrame({"date": pd.date_range("2025-01-01", periods=200)})
fitness["day_of_week"] = fitness["date"].dt.day_name()

avg_steps = {"Monday": 8000, "Tuesday": 7500, "Wednesday": 8000, "Thursday": 7500,
             "Friday": 7000, "Saturday": 10500, "Sunday": 5500}
steps = fitness["day_of_week"].map(avg_steps) + rng.normal(0, 1500, 200)
steps[[45, 118, 160]] = [21500, 19800, 600]
minutes = steps / 200 + rng.normal(0, 8, 200)

fitness["steps"] = steps.round().astype(int)
fitness["calories_burned"] = (1600 + steps * 0.07 + minutes * 3 + rng.normal(0, 120, 200)).round().astype(int)
fitness["exercise_minutes"] = minutes.clip(0).round().astype(int)
fitness["active_hours"] = (steps / 4000 + rng.normal(0, 0.4, 200)).clip(0.2).round(1)

fitness.to_csv("fitness_data.csv", index=False)
print("Saved fitness_data.csv:", fitness.shape)

## 2. Load and inspect the data

In [ ]:
df = pd.read_csv("fitness_data.csv")
df.head()

In [ ]:
df.info()
print("Shape:", df.shape)

## 3. Descriptive statistics

In [ ]:
metrics = ["steps", "calories_burned", "exercise_minutes"]
df[metrics].agg(["mean", "median", "std", "min", "max"]).round(1)

## 4. Average activity by day of week

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = df.groupby("day_of_week")[metrics].mean().reindex(day_order).round(1)
by_day

## 5. Correlation

In [ ]:
pearson = df["steps"].corr(df["calories_burned"])
print("Pearson correlation (steps vs calories_burned):", round(pearson, 2))
df[["steps", "calories_burned", "exercise_minutes", "active_hours"]].corr().round(2)

## 6. Charts

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(by_day.index, by_day["steps"], color="steelblue")
plt.title("Average Steps by Day of Week")
plt.xlabel("Day of week")
plt.ylabel("Average steps")
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(df["steps"], df["calories_burned"], alpha=0.6)
plt.title("Steps vs Calories Burned")
plt.xlabel("steps")
plt.ylabel("calories_burned")
plt.show()

## 7. Outliers in steps (IQR method)

In [ ]:
q1 = df["steps"].quantile(0.25)
q3 = df["steps"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(f"Q1: {q1:,.0f} | Q3: {q3:,.0f} | IQR: {iqr:,.0f}")
print(f"Normal range: {lower:,.0f} to {upper:,.0f} steps")

outliers = df[(df["steps"] < lower) | (df["steps"] > upper)]
print("Outliers found:", len(outliers))
outliers

## 8. Conclusions
- **Most active day:** Saturday, with about 10,988 steps on average. Sunday is the least active, with about 5,210.
- **Correlation:** steps and calories_burned have a strong positive Pearson correlation of 0.85. More steps usually means more calories burned.
- **Outliers:** the IQR method (normal range 1,933 to 13,213 steps) found 3 outlier days: 21,500 steps on 2025-02-15, 19,800 on 2025-04-29, and only 600 on 2025-06-10.